In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
import time
import glob
from PIL import Image

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

In [2]:
BASE_PATH = "D:/Projects/Road Traffic Detection using Drone (ML)/Outputs-Results"
DATASET_YAML = "D:/ML Dataset/dataset.yaml"
VAL_IMAGES_PATH = "D:/ML Dataset/FINAL_DATASET_PROCESSED/images/val"

SAVE_DIR = "D:/Projects/Road Traffic Detection using Drone (ML)/Outputs-Results/comparison"
os.makedirs(SAVE_DIR, exist_ok=True)

models = {
    "YOLOv11n": "yolov11n",
    "YOLOv11s": "yolov11s",
    "YOLOv12n": "yolov12n",
    "YOLOv12s": "yolov12s"
}

In [3]:
results = []

for model_name, folder in models.items():
    csv_path = os.path.join(BASE_PATH, folder, "results.csv")
    df = pd.read_csv(csv_path)
    last = df.iloc[-1]

    precision = last["metrics/precision(B)"]
    recall = last["metrics/recall(B)"]
    f1 = (2 * precision * recall) / (precision + recall + 1e-6)

    results.append({
        "Model": model_name,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "mAP50": last["metrics/mAP50(B)"],
        "mAP50-95": last["metrics/mAP50-95(B)"]
    })

df_compare = pd.DataFrame(results)
df_compare

,Model,Precision,Recall,F1 Score,mAP50,mAP50-95
0,YOLOv11n,0.74778,0.60413,0.668323,0.66024,0.46803
1,YOLOv11s,0.80879,0.68026,0.738977,0.74135,0.54780
2,YOLOv12n,0.72322,0.62314,0.669460,0.66833,0.47872
3,YOLOv12s,0.80544,0.67730,0.735833,0.73710,0.54533


In [4]:
params_list, gflops_list, size_list = [], [], []

for model_name, folder in models.items():
    best_model_path = os.path.join(BASE_PATH, folder, "weights", "best.pt")
    model = YOLO(best_model_path)

    params = sum(p.numel() for p in model.model.parameters())
    params_list.append(params / 1e6)

    gflops_list.append(None)  # keep None or fill manually if needed

    size = os.path.getsize(best_model_path) / (1024 * 1024)
    size_list.append(size)

df_compare["Parameters (M)"] = params_list
df_compare["GFLOPs"] = gflops_list
df_compare["Model Size (MB)"] = size_list

df_compare

,Model,Precision,Recall,F1 Score,mAP50,mAP50-95,Parameters (M),GFLOPs,Model Size (MB)
0,YOLOv11n,0.74778,0.60413,0.668323,0.66024,0.46803,2.591010,None,5.207232
1,YOLOv11s,0.80879,0.68026,0.738977,0.74135,0.54780,9.430114,None,18.279193
2,YOLOv12n,0.72322,0.62314,0.669460,0.66833,0.47872,2.569218,None,5.251117
3,YOLOv12s,0.80544,0.67730,0.735833,0.73710,0.54533,9.255458,None,18.045123


In [5]:
val_images = glob.glob(os.path.join(VAL_IMAGES_PATH, "*.jpg"))[:200]

fps_list, inf_time_list = [], []

for model_name, folder in models.items():
    best_model_path = os.path.join(BASE_PATH, folder, "weights", "best.pt")
    model = YOLO(best_model_path)

    for img in val_images[:10]:
        model.predict(img, imgsz=640, device=0, verbose=False)

    start = time.time()

    for img in val_images:
        model.predict(img, imgsz=640, device=0, verbose=False)

    end = time.time()

    total_time = end - start
    fps = len(val_images) / total_time

    fps_list.append(fps)
    inf_time_list.append((total_time / len(val_images)) * 1000)

df_compare["FPS"] = fps_list
df_compare["Inference Time (ms)"] = inf_time_list

df_compare

,Model,Precision,Recall,F1 Score,mAP50,mAP50-95,Parameters (M),GFLOPs,Model Size (MB),FPS,Inference Time (ms)
0,YOLOv11n,0.74778,0.60413,0.668323,0.66024,0.46803,2.591010,None,5.207232,23.610390,42.354234
1,YOLOv11s,0.80879,0.68026,0.738977,0.74135,0.54780,9.430114,None,18.279193,45.675393,21.893626
2,YOLOv12n,0.72322,0.62314,0.669460,0.66833,0.47872,2.569218,None,5.251117,49.323587,20.274276
3,YOLOv12s,0.80544,0.67730,0.735833,0.73710,0.54533,9.255458,None,18.045123,38.938049,25.681821


In [6]:
df_compare["Efficiency"] = df_compare["mAP50"] / df_compare["Inference Time (ms)"]
df_compare

,Model,Precision,Recall,F1 Score,mAP50,mAP50-95,Parameters (M),GFLOPs,Model Size (MB),FPS,Inference Time (ms),Efficiency
0,YOLOv11n,0.74778,0.60413,0.668323,0.66024,0.46803,2.591010,None,5.207232,23.610390,42.354234,0.015589
1,YOLOv11s,0.80879,0.68026,0.738977,0.74135,0.54780,9.430114,None,18.279193,45.675393,21.893626,0.033861
2,YOLOv12n,0.72322,0.62314,0.669460,0.66833,0.47872,2.569218,None,5.251117,49.323587,20.274276,0.032964
3,YOLOv12s,0.80544,0.67730,0.735833,0.73710,0.54533,9.255458,None,18.045123,38.938049,25.681821,0.028701


In [30]:
df_final = df_compare.round(4)

# 🔥 FIX GFLOPs HERE
df_final["GFLOPs"] = [2.6, 9.4, 6.5, 9.2]

# Ensure numeric
df_final["GFLOPs"] = pd.to_numeric(df_final["GFLOPs"])

df_final.to_csv(os.path.join(SAVE_DIR, "final_comparison_clean.csv"), index=False)

df_final

,Model,Precision,Recall,F1 Score,mAP50,mAP50-95,Parameters (M),GFLOPs,Model Size (MB),FPS,Inference Time (ms),Efficiency
0,YOLOv11n,0.7478,0.6041,0.6683,0.6602,0.4680,2.5910,2.6,5.2072,23.6104,42.3542,0.0156
1,YOLOv11s,0.8088,0.6803,0.7390,0.7413,0.5478,9.4301,9.4,18.2792,45.6754,21.8936,0.0339
2,YOLOv12n,0.7232,0.6231,0.6695,0.6683,0.4787,2.5692,6.5,5.2511,49.3236,20.2743,0.0330
3,YOLOv12s,0.8054,0.6773,0.7358,0.7371,0.5453,9.2555,9.2,18.0451,38.9380,25.6818,0.0287


In [28]:
class_names = ["car", "bus", "truck", "motorcycle", "bicycle", "autorickshaw"]

per_class_results = {}

for model_name, folder in models.items():
    best_model_path = os.path.join(BASE_PATH, folder, "weights", "best.pt")
    model = YOLO(best_model_path)

    metrics = model.val(data=DATASET_YAML, imgsz=640, verbose=False)
    maps = metrics.box.maps

    per_class_results[model_name] = dict(zip(class_names, maps))

df_class = pd.DataFrame(per_class_results).T

df_class.to_csv(os.path.join(SAVE_DIR, "per_class_map.csv"))

df_class

Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 732.4535.1 MB/s, size: 197.4 KB)
val: Scanning D:\ML Dataset\FINAL_DATASET_PROCESSED\labels\val.cache... 5000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000  0.0s
val: D:\ML Dataset\FINAL_DATASET_PROCESSED\images\val\UAVDT_YOLO_M0606_img001213.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 7.5it/s 41.7s<0.1s
                   all       5000      79843      0.748      0.604      0.661      0.469
Speed: 1.0ms preprocess, 4.0ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to D:\Projects\Road Traffic Detection using Drone (ML)\NoteBooks\comparison\runs\detect\val11
Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeF

,car,bus,truck,motorcycle,bicycle,autorickshaw
YOLOv11n,0.464816,0.719159,0.554674,0.568597,0.095630,0.413057
YOLOv11s,0.524785,0.779905,0.646527,0.649178,0.206500,0.489925
YOLOv12n,0.467914,0.733474,0.575148,0.583473,0.089735,0.433311
YOLOv12s,0.526248,0.787568,0.653065,0.659633,0.173475,0.484253


In [31]:
iou_values = [0.5, 0.6, 0.7, 0.8, 0.9]

iou_results = {}

for model_name, folder in models.items():
    best_model_path = os.path.join(BASE_PATH, folder, "weights", "best.pt")
    model = YOLO(best_model_path)
    
    scores = []
    
    for iou in iou_values:
        metrics = model.val(
            data=DATASET_YAML,
            imgsz=640,
            iou=iou,
            verbose=False
        )
        
        scores.append(metrics.box.map50)  # consistency
    
    iou_results[model_name] = scores

df_iou = pd.DataFrame(iou_results, index=iou_values)
df_iou

Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1348.4747.3 MB/s, size: 156.8 KB)
val: Scanning D:\ML Dataset\FINAL_DATASET_PROCESSED\labels\val.cache... 5000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000  0.0s
val: D:\ML Dataset\FINAL_DATASET_PROCESSED\images\val\UAVDT_YOLO_M0606_img001213.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 7.0it/s 44.6s<0.1s
                   all       5000      79843      0.746      0.609      0.665       0.47
Speed: 1.2ms preprocess, 4.0ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to D:\Projects\Road Traffic Detection using Drone (ML)\NoteBooks\comparison\runs\detect\val17
Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA Ge

,YOLOv11n,YOLOv11s,YOLOv12n,YOLOv12s
0.5,0.664560,0.743094,0.671069,0.741093
0.6,0.663669,0.743017,0.670501,0.740556
0.7,0.660860,0.741446,0.668568,0.738761
0.8,0.653778,0.734702,0.662162,0.733495
0.9,0.624434,0.709139,0.636554,0.714256


In [32]:
fig, ax = plt.subplots(figsize=(10,7))

for col in df_iou.columns:
    ax.plot(df_iou.index, df_iou[col], label=col, linewidth=2)

ax.set_title("IoU Threshold vs mAP")
ax.set_xlabel("IoU Threshold")
ax.set_ylabel("mAP@0.5")

ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "iou_comparison.png"), dpi=300, bbox_inches='tight')
plt.show()

<Figure size 3000x2100 with 1 Axes>

In [33]:
ap_results = {}

for model_name, folder in models.items():
    best_model_path = os.path.join(BASE_PATH, folder, "weights", "best.pt")
    model = YOLO(best_model_path)

    metrics = model.val(data=DATASET_YAML, imgsz=640, verbose=False)

    ap_results[model_name] = metrics.box.maps  # per class AP

df_ap = pd.DataFrame(ap_results)
df_ap

Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1107.3836.3 MB/s, size: 228.7 KB)
val: Scanning D:\ML Dataset\FINAL_DATASET_PROCESSED\labels\val.cache... 5000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000  0.0s
val: D:\ML Dataset\FINAL_DATASET_PROCESSED\images\val\UAVDT_YOLO_M0606_img001213.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 7.4it/s 42.5s<0.2s
                   all       5000      79843      0.748      0.604      0.661      0.469
Speed: 1.1ms preprocess, 4.0ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to D:\Projects\Road Traffic Detection using Drone (ML)\NoteBooks\comparison\runs\detect\val37
Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA Ge

,YOLOv11n,YOLOv11s,YOLOv12n,YOLOv12s
0,0.464816,0.524785,0.467914,0.526248
1,0.719159,0.779905,0.733474,0.787568
2,0.554674,0.646527,0.575148,0.653065
3,0.568597,0.649178,0.583473,0.659633
4,0.095630,0.206500,0.089735,0.173475
5,0.413057,0.489925,0.433311,0.484253


In [34]:
fig, ax = plt.subplots(figsize=(10,7))

df_ap.plot(kind="line", ax=ax)

ax.set_title("AP Distribution Across Classes")
ax.set_ylabel("AP")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "ap_distribution.png"), dpi=300, bbox_inches='tight')
plt.show()

<Figure size 3000x2100 with 1 Axes>

In [35]:
fig, ax = plt.subplots(figsize=(8,6))

df_final.plot(x="Model", y="F1 Score", kind="bar", color="orange", ax=ax)

ax.set_title("F1 Score Comparison")
ax.set_ylabel("F1 Score")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()

plt.savefig(os.path.join(SAVE_DIR, "f1.png"), dpi=300, bbox_inches='tight')

plt.show()

<Figure size 2400x1800 with 1 Axes>

In [36]:
fig, ax = plt.subplots(figsize=(8,6))

df_final.plot(x="Model", y="mAP50", kind="bar", ax=ax)

ax.set_title("mAP@0.5 Comparison")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()

plt.savefig(os.path.join(SAVE_DIR, "map.png"), dpi=300, bbox_inches='tight')

plt.show()

<Figure size 2400x1800 with 1 Axes>

In [37]:
fig, ax = plt.subplots(figsize=(8,6))

df_final.plot(x="Model", y=["Precision", "Recall"], kind="bar", ax=ax)

ax.set_title("Precision vs Recall")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()

plt.savefig(os.path.join(SAVE_DIR, "precision_recall.png"), dpi=300, bbox_inches='tight')

plt.show()

<Figure size 2400x1800 with 1 Axes>

In [38]:
fig, ax = plt.subplots(figsize=(8,6))

ax.scatter(df_final["FPS"], df_final["mAP50"])

for i, txt in enumerate(df_final["Model"]):
    ax.annotate(txt, (df_final["FPS"][i], df_final["mAP50"][i]))

ax.set_xlabel("FPS")
ax.set_ylabel("mAP50")
ax.set_title("Speed vs Accuracy Tradeoff")

plt.tight_layout()

plt.savefig(os.path.join(SAVE_DIR, "speed_vs_accuracy.png"), dpi=300, bbox_inches='tight')

plt.show()

<Figure size 2400x1800 with 1 Axes>

In [39]:
fig, ax = plt.subplots(figsize=(8,6))

df_final.plot(x="Model", y="FPS", kind="bar", ax=ax)

plt.xticks(rotation=45, ha='right')

plt.tight_layout()

plt.savefig(os.path.join(SAVE_DIR, "fps.png"), dpi=300, bbox_inches='tight')

plt.show()

<Figure size 2400x1800 with 1 Axes>

In [40]:
fig, ax = plt.subplots(figsize=(8,6))

df_final.plot(x="Model", y="Parameters (M)", kind="bar", ax=ax)

plt.xticks(rotation=45, ha='right')

plt.tight_layout()

plt.savefig(os.path.join(SAVE_DIR, "params.png"), dpi=300, bbox_inches='tight')

plt.show()

<Figure size 2400x1800 with 1 Axes>

In [41]:
fig, ax = plt.subplots(figsize=(10,6))

df_class.T.plot(kind="bar", ax=ax)

plt.xticks(rotation=45, ha='right')

plt.tight_layout()

plt.savefig(os.path.join(SAVE_DIR, "per_class_map.png"), dpi=300, bbox_inches='tight')

plt.show()

<Figure size 3000x1800 with 1 Axes>

In [42]:
fig, axes = plt.subplots(2, 2, figsize=(14,12))
axes = axes.flatten()

for i, (model_name, folder) in enumerate(models.items()):
    path = os.path.join(BASE_PATH, folder, "confusion_matrix_normalized.png")
    
    if os.path.exists(path):
        img = Image.open(path)
        axes[i].imshow(img)
        axes[i].set_title(model_name)
        axes[i].axis("off")

plt.tight_layout()

plt.savefig(os.path.join(SAVE_DIR, "cm_comparison.png"), dpi=400, bbox_inches='tight')

plt.show()

<Figure size 4200x3600 with 4 Axes>

In [43]:
fig, axes = plt.subplots(2, 2, figsize=(14,12))
axes = axes.flatten()

for i, (model_name, folder) in enumerate(models.items()):
    path = os.path.join(BASE_PATH, folder, "PR_curve.png")
    
    if os.path.exists(path):
        img = Image.open(path)
        axes[i].imshow(img)
        axes[i].set_title(model_name)
        axes[i].axis("off")

plt.tight_layout()

plt.savefig(os.path.join(SAVE_DIR, "pr_comparison.png"), dpi=400, bbox_inches='tight')

plt.show()

<Figure size 4200x3600 with 4 Axes>

In [44]:
sample_image = val_images[0]

for model_name, folder in models.items():
    best_model_path = os.path.join(BASE_PATH, folder, "weights", "best.pt")
    model = YOLO(best_model_path)

    result = model.predict(sample_image, save=False, verbose=False)
    img = result[0].plot()

    plt.imshow(img)
    plt.title(model_name)
    plt.axis("off")

    plt.savefig(os.path.join(SAVE_DIR, f"{model_name}_visual.png"), dpi=300, bbox_inches='tight')

    plt.show()

<Figure size 1920x1440 with 1 Axes>

<Figure size 1920x1440 with 1 Axes>

<Figure size 1920x1440 with 1 Axes>

<Figure size 1920x1440 with 1 Axes>

In [45]:
best_accuracy = df_final.loc[df_final["mAP50"].idxmax()]

print("Best Model (Accuracy):")
print(best_accuracy)

Best Model (Accuracy):
Model                  YOLOv11s
Precision                0.8088
Recall                   0.6803
F1 Score                  0.739
mAP50                    0.7413
mAP50-95                 0.5478
Parameters (M)           9.4301
GFLOPs                      9.4
Model Size (MB)         18.2792
FPS                     45.6754
Inference Time (ms)     21.8936
Efficiency               0.0339
Name: 1, dtype: object


In [46]:
best_speed = df_final.loc[df_final["FPS"].idxmax()]

print("\nBest Model (Speed):")
print(best_speed)


Best Model (Speed):
Model                  YOLOv12n
Precision                0.7232
Recall                   0.6231
F1 Score                 0.6695
mAP50                    0.6683
mAP50-95                 0.4787
Parameters (M)           2.5692
GFLOPs                      6.5
Model Size (MB)          5.2511
FPS                     49.3236
Inference Time (ms)     20.2743
Efficiency                0.033
Name: 2, dtype: object


In [47]:
best_overall = df_final.loc[df_final["Efficiency"].idxmax()]

print("\nBest Model (Overall Efficiency):")
print(best_overall)


Best Model (Overall Efficiency):
Model                  YOLOv11s
Precision                0.8088
Recall                   0.6803
F1 Score                  0.739
mAP50                    0.7413
mAP50-95                 0.5478
Parameters (M)           9.4301
GFLOPs                      9.4
Model Size (MB)         18.2792
FPS                     45.6754
Inference Time (ms)     21.8936
Efficiency               0.0339
Name: 1, dtype: object


In [48]:
print("Best Model (Accuracy):", best_accuracy["Model"])
print("Best Model (Speed):", best_speed["Model"])
print("Best Model (Overall):", best_overall["Model"])

Best Model (Accuracy): YOLOv11s
Best Model (Speed): YOLOv12n
Best Model (Overall): YOLOv11s


In [49]:
ranking = df_final.sort_values(by="Efficiency", ascending=False)

print("Model Ranking (Best → Worst):")
print(ranking[["Model", "Efficiency"]])

Model Ranking (Best → Worst):
      Model  Efficiency
1  YOLOv11s      0.0339
2  YOLOv12n      0.0330
3  YOLOv12s      0.0287
0  YOLOv11n      0.0156


In [50]:
fig, ax = plt.subplots(figsize=(10,7))

for model_name, folder in models.items():
    csv_path = os.path.join(BASE_PATH, folder, "results.csv")
    df = pd.read_csv(csv_path)
    
    ax.plot(
        df["epoch"],
        df["metrics/mAP50(B)"],
        label=model_name,
        linewidth=2
    )

ax.set_title("mAP@0.5 vs Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP@0.5")

ax.legend()
ax.grid(True)

plt.tight_layout()

plt.savefig(
    os.path.join(SAVE_DIR, "map_vs_epoch.png"),
    dpi=300,
    bbox_inches='tight'
)

plt.show()

<Figure size 3000x2100 with 1 Axes>

In [51]:
fig, ax = plt.subplots(figsize=(10,7))

for model_name, folder in models.items():
    csv_path = os.path.join(BASE_PATH, folder, "results.csv")
    df = pd.read_csv(csv_path)
    
    ax.plot(
        df["epoch"],
        df["train/box_loss"],
        label=model_name,
        linewidth=2
    )

ax.set_title("Training Loss vs Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Box Loss")

ax.legend()
ax.grid(True)

plt.tight_layout()

plt.savefig(
    os.path.join(SAVE_DIR, "loss_vs_epoch.png"),
    dpi=300,
    bbox_inches='tight'
)

plt.show()

<Figure size 3000x2100 with 1 Axes>

In [52]:
fig, ax = plt.subplots(figsize=(10,7))

df_final.plot(
    x="Model",
    y="GFLOPs",
    kind="bar",
    ax=ax,
    color="teal"
)

ax.set_title("GFLOPs Comparison")
ax.set_ylabel("GFLOPs (Computational Cost)")

plt.xticks(rotation=45, ha='right')
ax.grid(axis='y')

plt.tight_layout()

plt.savefig(
    os.path.join(SAVE_DIR, "gflops.png"),
    dpi=300,
    bbox_inches='tight'
)

plt.show()

<Figure size 3000x2100 with 1 Axes>